# SHAPY deployment notebook — VS Code + WSL host + Docker runtime

This notebook is intended to run with the **Python 3.8 (SHAPY Docker)** kernel exposed by the Docker container.

Design:

- VS Code / WSL host: Ubuntu 20.04 LTS.
- SHAPY runtime: Docker container using Ubuntu 18.04 + CUDA 10.2.
- Main workflow: install/verify SHAPY, run the official regressor demo, compute virtual measurements, and optionally evaluate on HBW.

In [ ]:
# CELL 1: Verify that this notebook is running inside the SHAPY Docker runtime.

import os
import sys
import platform
import subprocess
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("SHAPY_DIR:", os.environ.get("SHAPY_DIR"))
print("PYOPENGL_PLATFORM:", os.environ.get("PYOPENGL_PLATFORM"))

try:
    import torch
    print("Torch version:", torch.__version__)
    print("Torch CUDA runtime:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch check failed:", repr(exc))

print("\nNVIDIA-SMI:")
subprocess.run("nvidia-smi", shell=True, check=False)

In [ ]:
# CELL 2: Define project paths and reusable helpers.

import os
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

SHAPY_DIR = Path(os.environ.get("SHAPY_DIR", "/workspace/shapy")).resolve()
DATA_DIR = SHAPY_DIR / "data"
SAMPLES_DIR = SHAPY_DIR / "samples"
OUTPUT_DIR = SAMPLES_DIR / "shapy_fit"
VM_OUTPUT_DIR = SAMPLES_DIR / "virtual_measurements"
REPORT_DIR = SHAPY_DIR / "deployment_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["SHAPY_DIR"] = str(SHAPY_DIR)
os.environ["PYTHONPATH"] = f"{SHAPY_DIR}:{SHAPY_DIR / 'attributes'}:" + os.environ.get("PYTHONPATH", "")
os.environ["CUDA_HOME"] = os.environ.get("CUDA_HOME", "/usr/local/cuda")
os.environ["PYOPENGL_PLATFORM"] = os.environ.get("PYOPENGL_PLATFORM", "egl")

def run(cmd, cwd=None, check=True):
    """Run a shell command and print it before execution."""
    print(f"\n$ {cmd}\n")
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        shell=True,
        check=check,
        text=True
    )

print("SHAPY_DIR:", SHAPY_DIR)
print("DATA_DIR:", DATA_DIR)
print("SAMPLES_DIR:", SAMPLES_DIR)

In [ ]:
# CELL 3: Install SHAPY dependencies and local CUDA/C++ extensions inside the container.

# This cell is idempotent: it can be re-run after rebuilding the container.
# It uses the helper script mounted from the package if available.
# Expected duration: several minutes, mainly because of pip packages and CUDA extension compilation.

INSTALL_SCRIPT_CANDIDATES = [
    Path("/workspace/scripts/install_shapy_inside_container.sh"),
    Path("/workspace/../scripts/install_shapy_inside_container.sh"),
    Path("/workspace/shapy_docker_vscode_package/scripts/install_shapy_inside_container.sh"),
]

script = next((p for p in INSTALL_SCRIPT_CANDIDATES if p.exists()), None)

if script is None:
    # Fallback: run the same installation steps directly from the notebook.
    install_cmd = r"""
    set -euo pipefail
    export SHAPY_DIR="${SHAPY_DIR:-/workspace/shapy}"
    export PYTHONPATH="${SHAPY_DIR}:${SHAPY_DIR}/attributes:${PYTHONPATH:-}"
    export CUDA_HOME="${CUDA_HOME:-/usr/local/cuda}"
    export FORCE_CUDA=1

    cd "${SHAPY_DIR}"
    python -m pip install --upgrade "pip<24" "setuptools<60" wheel ninja cython
    grep -vE '^(torch|torchvision)==|^sklearn==|^open3d$' requirements.txt > /tmp/shapy_requirements_filtered.txt
    python -m pip install -r /tmp/shapy_requirements_filtered.txt
    python -m pip install "open3d==0.17.0" "pyrender==0.1.43"

    cd "${SHAPY_DIR}/attributes"
    python setup.py install

    cd "${SHAPY_DIR}/mesh-mesh-intersection"
    export CUDA_SAMPLES_INC="$(pwd)/include"
    python -m pip install -r requirements.txt
    python setup.py install
    """
    run(install_cmd)
else:
    run(f"bash {script}")

In [ ]:
# CELL 4: Verify the required SHAPY data and SMPL/SMPL-X model folder structure.

required_paths = [
    DATA_DIR / "body_models" / "smplx" / "SMPLX_NEUTRAL.npz",
    DATA_DIR / "body_models" / "smplx" / "SMPLX_FEMALE.npz",
    DATA_DIR / "body_models" / "smplx" / "SMPLX_MALE.npz",
    DATA_DIR / "trained_models" / "shapy" / "SHAPY_A",
    DATA_DIR / "expose_release",
    DATA_DIR / "utility_files",
]

optional_paths = [
    DATA_DIR / "body_models" / "smpl" / "SMPL_NEUTRAL.pkl",
    DATA_DIR / "body_models" / "smpl" / "SMPL_FEMALE.pkl",
    DATA_DIR / "body_models" / "smpl" / "SMPL_MALE.pkl",
]

print("Required paths:")
missing_required = []
for p in required_paths:
    ok = p.exists()
    print(f"{'OK     ' if ok else 'MISSING'} {p}")
    if not ok:
        missing_required.append(str(p))

print("\nOptional SMPL paths:")
for p in optional_paths:
    print(f"{'OK     ' if p.exists() else 'MISSING'} {p}")

if missing_required:
    print("\nACTION REQUIRED:")
    print("Download SHAPY data and SMPL-X files according to the licenses, then place them in the paths above.")
    print("You can run the official SHAPY downloader from the next cell if you have SHAPY website credentials.")
else:
    print("\nAll required SHAPY paths are present.")

In [ ]:
# CELL 5: Optional official SHAPY data download after registration.

# Use this only if:
# 1) you have registered on the official SHAPY website,
# 2) you accept the license terms,
# 3) you want the script to prompt for username/password inside the notebook terminal output.

download_script = DATA_DIR / "download_data.sh"

if not download_script.exists():
    print("download_data.sh was not found. Check that the official SHAPY repository is cloned correctly.")
else:
    print("This command is intentionally not executed automatically because it asks for credentials.")
    print("To run it, uncomment the next line:")
    print(f"cd {DATA_DIR} && bash download_data.sh")
    # run("bash download_data.sh", cwd=DATA_DIR)

In [ ]:
# CELL 6: Prepare sample input folders for the official SHAPY regressor demo.

# The official demo expects:
#   samples/images/    -> input images
#   samples/openpose/  -> OpenPose keypoints
#
# The repository normally includes demo samples. This cell lists what is available.

img_dir = SAMPLES_DIR / "images"
keyp_dir = SAMPLES_DIR / "openpose"

img_dir.mkdir(parents=True, exist_ok=True)
keyp_dir.mkdir(parents=True, exist_ok=True)

image_files = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
keypoint_files = sorted(keyp_dir.glob("*.json"))

print(f"Images found: {len(image_files)}")
for p in image_files[:10]:
    print("  ", p.name)

print(f"\nOpenPose JSON files found: {len(keypoint_files)}")
for p in keypoint_files[:10]:
    print("  ", p.name)

if len(image_files) == 0 or len(keypoint_files) == 0:
    print("\nACTION REQUIRED:")
    print("Add demo images to samples/images and matching OpenPose JSON files to samples/openpose.")
    print("If the official sample data is present, this should already be populated.")

In [ ]:
# CELL 7: Run the official SHAPY regressor demo on the sample images.

# This is the core deployment smoke test.
# It saves visualizations, parameters, and meshes into samples/shapy_fit.

demo_cmd = r"""
python demo.py \
  --save-vis true \
  --save-params true \
  --save-mesh true \
  --split test \
  --datasets openpose \
  --output-folder ../samples/shapy_fit/ \
  --exp-cfg configs/b2a_expose_hrnet_demo.yaml \
  --exp-opts \
    output_folder=../data/trained_models/shapy/SHAPY_A \
    part_key=pose \
    datasets.pose.openpose.data_folder=../samples \
    datasets.pose.openpose.img_folder=images \
    datasets.pose.openpose.keyp_folder=openpose \
    datasets.batch_size=1 \
    datasets.pose_shape_ratio=1.0
"""

run(demo_cmd, cwd=SHAPY_DIR / "regressor")

In [ ]:
# CELL 8: Inspect SHAPY output files after the regressor demo.

from pathlib import Path
import os

if not OUTPUT_DIR.exists():
    print("Output folder does not exist yet:", OUTPUT_DIR)
else:
    all_outputs = sorted([p for p in OUTPUT_DIR.rglob("*") if p.is_file()])
    print(f"Number of output files: {len(all_outputs)}")
    for p in all_outputs[:50]:
        rel = p.relative_to(OUTPUT_DIR)
        size_kb = p.stat().st_size / 1024
        print(f"{rel}  ({size_kb:.1f} KB)")

    mesh_files = [p for p in all_outputs if p.suffix.lower() in [".obj", ".ply"]]
    param_files = [p for p in all_outputs if p.suffix.lower() in [".pkl", ".npz"]]
    vis_files = [p for p in all_outputs if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]

    print("\nSummary:")
    print("Meshes:", len(mesh_files))
    print("Parameter files:", len(param_files))
    print("Visualizations:", len(vis_files))

In [ ]:
# CELL 9: Compute virtual anthropometric measurements from SHAPY outputs.

# The official virtual_measurements.py script expects the input folder to contain
# SHAPY-style saved outputs. The repository also includes sample folders for this.
#
# If your regressor output is not accepted by the script, use the official sample:
#   ../samples/shapy_fit_for_virtual_measurements/

candidate_inputs = [
    OUTPUT_DIR,
    SAMPLES_DIR / "shapy_fit_for_virtual_measurements",
]

input_folder = next((p for p in candidate_inputs if p.exists()), None)

if input_folder is None:
    print("No valid input folder found for virtual measurements.")
else:
    print("Using input folder:", input_folder)
    VM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    vm_cmd = f"""
    python virtual_measurements.py \
      --input-folder {input_folder} \
      --output-folder {VM_OUTPUT_DIR}
    """
    run(vm_cmd, cwd=SHAPY_DIR / "measurements", check=False)

In [ ]:
# CELL 10: Inspect virtual measurement outputs.

if not VM_OUTPUT_DIR.exists():
    print("Virtual-measurement output folder does not exist yet:", VM_OUTPUT_DIR)
else:
    files = sorted([p for p in VM_OUTPUT_DIR.rglob("*") if p.is_file()])
    print(f"Virtual-measurement files: {len(files)}")
    for p in files[:50]:
        print(" ", p.relative_to(VM_OUTPUT_DIR))

    # Try to preview simple CSV/TXT/JSON-like outputs if they exist.
    for p in files:
        if p.suffix.lower() in [".txt", ".csv", ".json", ".yaml", ".yml"]:
            print(f"\n--- Preview: {p.name} ---")
            print(p.read_text(errors="ignore")[:3000])
            break

In [ ]:
# CELL 11: Optional HBW validation evaluation for SHAPY.

# Before running this cell:
#   1) Download HBW according to its access rules.
#   2) Place it somewhere accessible, for example /workspace/datasets/HBW.
#   3) Create a symlink:
#        ln -s /workspace/datasets/HBW /workspace/shapy/datasets/HBW
#
# This command follows the official SHAPY evaluation entry point.

HBW_LINK = SHAPY_DIR / "datasets" / "HBW"

if not HBW_LINK.exists():
    print("HBW dataset link not found:", HBW_LINK)
    print("Skipping HBW evaluation.")
else:
    eval_cmd = r"""
    python evaluate.py \
      --exp-cfg configs/b2a_expose_hrnet_eval_shape.yaml \
      --exp-opts \
        output_folder=../data/trained_models/shapy/SHAPY_A \
        datasets.batch_size=1 \
        datasets.pose_shape_ratio=0.0 \
        is_training=False \
        run_final_evaluation_on_validation_set=True
    """
    run(eval_cmd, cwd=SHAPY_DIR / "regressor")

In [ ]:
# CELL 12: Save a reproducibility/deployment report for the final project documentation.

import json
import subprocess
from datetime import datetime

def command_output(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except subprocess.CalledProcessError as exc:
        return exc.output.strip()

report = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "method": "SHAPY",
    "host_design": "VS Code connected to WSL Ubuntu 20.04; runtime isolated in Docker",
    "runtime_design": "Docker Ubuntu 18.04 + CUDA 10.2 + Python 3.8 + PyTorch 1.7.1/cu102",
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "torch_version": None,
    "torch_cuda": None,
    "cuda_available": None,
    "gpu": None,
    "shapy_dir": str(SHAPY_DIR),
    "data_dir": str(DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "virtual_measurement_output_dir": str(VM_OUTPUT_DIR),
    "git_commit": command_output(f"cd {SHAPY_DIR} && git rev-parse HEAD"),
    "git_status": command_output(f"cd {SHAPY_DIR} && git status --short"),
    "nvidia_smi": command_output("nvidia-smi"),
    "pip_freeze": command_output("python -m pip freeze"),
}

try:
    import torch
    report["torch_version"] = torch.__version__
    report["torch_cuda"] = torch.version.cuda
    report["cuda_available"] = torch.cuda.is_available()
    report["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception as exc:
    report["torch_error"] = repr(exc)

report_path = REPORT_DIR / f"shapy_deployment_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
report_path.write_text(json.dumps(report, indent=2))
print("Saved report:", report_path)
print(json.dumps({k: report[k] for k in ["method", "runtime_design", "torch_version", "torch_cuda", "cuda_available", "gpu", "git_commit"]}, indent=2))

In [ ]:
# CELL 13: Minimal metric helpers for the final comparison tables.

# These functions are independent from SHAPY and can be reused to evaluate predicted
# joints/vertices against ground truth arrays when you have aligned data.

import numpy as np

def mpjpe(pred_joints, gt_joints):
    """
    Mean Per-Joint Position Error.
    pred_joints, gt_joints: arrays shaped (..., J, 3)
    returns mean Euclidean error.
    """
    pred_joints = np.asarray(pred_joints)
    gt_joints = np.asarray(gt_joints)
    return np.linalg.norm(pred_joints - gt_joints, axis=-1).mean()

def pve(pred_vertices, gt_vertices):
    """
    Per-Vertex Error / Mean Per-Vertex Position Error.
    pred_vertices, gt_vertices: arrays shaped (..., V, 3)
    """
    pred_vertices = np.asarray(pred_vertices)
    gt_vertices = np.asarray(gt_vertices)
    return np.linalg.norm(pred_vertices - gt_vertices, axis=-1).mean()

def beta_l2_error(pred_betas, gt_betas):
    """
    L2 error between predicted and ground-truth SMPL/SMPL-X shape parameters.
    pred_betas, gt_betas: arrays shaped (..., B)
    """
    pred_betas = np.asarray(pred_betas)
    gt_betas = np.asarray(gt_betas)
    return np.linalg.norm(pred_betas - gt_betas, axis=-1).mean()

def pck(pred_joints, gt_joints, threshold=0.15):
    """
    Percentage of Correct Keypoints.
    threshold is in the same unit as the joint coordinates, e.g. meters.
    """
    pred_joints = np.asarray(pred_joints)
    gt_joints = np.asarray(gt_joints)
    errors = np.linalg.norm(pred_joints - gt_joints, axis=-1)
    return (errors < threshold).mean()

print("Metric helper functions loaded: mpjpe, pve, beta_l2_error, pck")